In [2]:
import pandas as pd
import numpy as np
import re
# step 1: Load and inspect the dataset
df = pd.read_csv("unclaimed_business_performance_dataset.csv")
print(df.head(5))
#inspect and understand the data
#shape(row,column)
print("shape: \n",df.shape)
print("First 10 rows: \n",df.head(10))
print("Last 10 rows: \n",df.tail(10))
#display the column names from the dataset
print("column names:",df.columns.tolist())
#display the datatype of the columns
print("Data type: \n",df.dtypes)
#display the summary information
print("Summary information: \n",df.info())
#display statistical summary only for number columns
print("statistical summary: \n",df.describe())
#display statistical summary of both both number and string column:
print("statistical summary for all: \n",df.describe(include='all'))
#Unique values per columns:
print("Unique values:",df.nunique())
#Values count of the column
print(df['Region'].value_counts())
#sample() select any random rows
print("Sample rows \n",df.sample(5))
#Detect missing values and count missing values per column
print("count of missing values: ",df.isnull().sum())
#percentage of missing column per columns
print("Percentage of missing values: \n",df.isnull().sum() / len(df) * 100)

   Transaction_ID      Order_Date   Region Product_Category   Product_Name  \
0          100521      18/08/2024     west         Clothing       Sneakers   
1          101911     16-Jul-2025  CENTRAL      ELECTRONICS         Laptop   
2          103248             NaN     EAST        Furniture    Bed Frame     
3          102550  04 August 2025     East           Cloths       Sneakers   
4          102100             NaN    north      Electronics     Smartwatch   

   Units_Sold Unit_Price  Total_Revenue Customer_Name Payment_Method  
0        34.0     4805.0      163370.00    Anjali Das           cash  
1        14.0     1845.6       25838.40  Suresh Joshi    CREDIT_CARD  
2        26.0    2565.15       66693.90  Anjali Reddy            upi  
3        26.0    3841.39       99876.14   Manoj Kumar     DEBIT-CARD  
4        37.0     254.22        9406.14    Ajay Gupta    credit card  
shape: 
 (4000, 10)
First 10 rows: 
    Transaction_ID      Order_Date      Region Product_Category   Pro

In [2]:
#step 2: Standardize column names
df = pd.read_csv("unclaimed_business_performance_dataset.csv")
df.columns = df.columns.str.strip().str.replace(" ","_")
             
print(df.columns.tolist())


['Transaction_ID', 'Order_Date', 'Region', 'Product_Category', 'Product_Name', 'Units_Sold', 'Unit_Price', 'Total_Revenue', 'Customer_Name', 'Payment_Method']


In [3]:
#step 3: Remove extact duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} exact duplicate rows")


Removed 59 exact duplicate rows


In [4]:
#step 4 Handle duplicate transcation_ids
df['Completeness'] = df.notna().sum(axis=1)
df = (
    df.sort_values('Completeness', ascending = False)
     .drop_duplicates(subset = 'Transaction_ID',keep = 'first')
     .drop(columns = 'Completeness')
     )
print("Shapes after resolving duplicate Transaction_IDs: ",df.shape)

Shapes after resolving duplicate Transaction_IDs:  (3864, 10)


In [5]:
#step 5 : Standardize text case in categorical columns
categorical_cols = ["Region", "Product_Category", "Product_Name", "Payment_Method"]
for col in categorical_cols:
    df[col] = df[col].astype('string').str.strip().str.title()

In [6]:
#step 6 Trim whitespace from all the text fields
text_cols = df.select_dtypes(include = 'string').columns
for col in text_cols:
    df[col] = df[col].str.strip()

    

In [7]:
print(df["Customer_Name"].isna().sum())
junk_in_name = ["none","null"]
df['Customer_Name'] = df['Customer_Name'].replace(junk_in_name,"Unknown")

144


In [8]:
#step 7  Fix inconsistent category labels
regional_map = {
    "Centeral" : "Central",
}
category_map = {
    "Cloths": "Clothing",
    "Electronic": "Electronics",
    "Furnitures": "Furniture",
    "Stationary": "Stationery",
    "Grocery": "Groceries",
}
payment_map = {
    "Cc" : "Credit Card",
    "Credit_Card": "Credit Card",
    "Debit-Card": "Debit Card",
    "Upi": "UPI",
    "Net Banking": "Net Banking",
    "Netbanking": "Net Banking",
}
df['Region'] = df['Region'].replace(regional_map)
df['Product_Category'] = df['Product_Category'].replace(category_map)
df['Payment_Method'] = df['Payment_Method'].replace(payment_map)


In [1]:
#step 8 replace placeholders junk values
junk_values = ["-", "?", "None", "Null", "N/A", "Na", "Nan", ""]
df = df.replace(junk_values,np.nan)

NameError: name 'df' is not defined

In [9]:
# step 9: 	Clean and convert Unit_Price
def clean_numeric(series):
    return pd.to_numeric(
        series.astype("string").str.replace(r"[\$,]", "", regex=True).str.strip(),
        errors="coerce",
    )

df["Units_Sold"] = clean_numeric(df["Units_Sold"])
df["Unit_Price"] = clean_numeric(df["Unit_Price"])
df["Total_Revenue"] = clean_numeric(df["Total_Revenue"])

In [10]:
#step 10 Handling missing values 
#categorical columns
for col in ["Region", "Product_Category", "Payment_Method", "Customer_Name"]:
    df[col] = df[col].fillna("Unknown")
#datetime column
df = df.dropna(subset= ["Order_Date"])
#Units_Sold / Unit_Price missing: these are numeric and central to the
# analysis (revenue = units x price), so we don't guess a fabricated number.
df["Missing_Numeric_Flag"] = df["Units_Sold"].isna() | df["Unit_Price"].isna()
print(f"Rows flagged for missing Unit_Sold or Unit_Price : {df['Missing_Numeric_Flag'].sum()} ")


Rows flagged for missing Unit_Sold or Unit_Price : 262 


In [11]:
#step 11 Impute missing Units_Sold / Unit_Price with the median
df["Units_Sold"] = df.groupby("Product_Category")["Units_Sold"].transform(
    lambda x: x.fillna(x.median())
)
df["Unit_Price"] = df.groupby("Product_Category")["Unit_Price"].transform(
    lambda x: x.fillna(x.median())
)
df["Units_Sold"] = df["Units_Sold"].fillna(df["Units_Sold"].median())
df["Unit_Price"] = df["Unit_Price"].fillna(df["Unit_Price"].median())
print("Remaining nulls in Units_Sold/Unit_Price after imputation:",
      df["Units_Sold"].isna().sum() + df["Unit_Price"].isna().sum())

Remaining nulls in Units_Sold/Unit_Price after imputation: 0


In [12]:
# step 12 Standardize order_date format
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce", format="mixed", dayfirst=True)
unparsed_dates = df["Order_Date"].isna().sum()
print(f"Rows where date could not be parsed: {unparsed_dates}")
if unparsed_dates > 0:
    df = df.dropna(subset=["Order_Date"])
    print(f"Dropped {unparsed_dates} rows with unparseable dates")

Rows where date could not be parsed: 0


In [13]:
#step 13 Fix invalid negative values in units_sold
df["Units_Sold"] = df["Units_Sold"].abs()

In [14]:
#step 14 Detect and treat outliers in unit price
Q1 = df["Unit_Price"].quantile(0.25)
Q3 = df["Unit_Price"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df["Unit_Price"] < lower_bound) | (df["Unit_Price"] > upper_bound)]
print(f"outliers detect in Unit_Price : {len(outliers)}")
df["Unit_Price"] = df["Unit_Price"].clip(lower=lower_bound, upper=upper_bound)

outliers detect in Unit_Price : 14


In [15]:
#step 15 Recalculate and validate total revenue
df["Total_Revenue_Calculated"]= (df["Units_Sold"] * df["Unit_Price"]).round(2)
df["Revenue_Mismatch_Flag"] =( (df["Total_Revenue"] - df["Total_Revenue_Calculated"]).abs() > 1)
print(df["Revenue_Mismatch_Flag"])
print(f"Rows with total_revenue mismatch: {df['Revenue_Mismatch_Flag'].sum()}")
print(df["Total_Revenue"])
df["Total_Revenue"] = df["Total_Revenue"].fillna(df["Total_Revenue_Calculated"])
print(df["Total_Revenue"])
df["Revenue_Mismatch_Flag"] = df["Revenue_Mismatch_Flag"].fillna(False)

0       False
2481    False
2445    False
2447    False
2448    False
        ...  
3844     <NA>
3181    False
2478     <NA>
2437     <NA>
222     False
Name: Revenue_Mismatch_Flag, Length: 3786, dtype: boolean
Rows with total_revenue mismatch: 402
0        163370.0
2481      89212.2
2445       1263.0
2447     22409.46
2448     10585.39
          ...    
3844         <NA>
3181    112375.01
2478         <NA>
2437         <NA>
222      16032.96
Name: Total_Revenue, Length: 3786, dtype: Float64
0        163370.0
2481      89212.2
2445       1263.0
2447     22409.46
2448     10585.39
          ...    
3844     53098.65
3181    112375.01
2478     10050.04
2437     74560.02
222      16032.96
Name: Total_Revenue, Length: 3786, dtype: Float64


In [16]:
#step 16 Standardize payment_method categories
print("Final Payment_method categories: ",df["Payment_Method"].unique())

Final Payment_method categories:  <StringArray>
[       'Cash', 'Credit Card', 'Net Banking',  'Debit Card',         'UPI',
           '?',           '-',        'None',     'Unknown']
Length: 9, dtype: string


In [17]:
#step 17 Final validate pass -  recheck
print("\n === FINAL VALIDATION ===")
print("Final Shape:",df.shape)
print("\nRemaining nulls per column:")
print(df.isnull().sum())
print("\n Data types: ")
print(df.dtypes)
print("\nRegional Categories: ",df['Region'].unique())
print("product_category :",df["Product_Category"].unique())
print("Payment_Method_Categories :",df["Payment_Method"].unique())
print("\n Units_Sold min/max :",df["Units_Sold"].min(), "/",df["Units_Sold"].max())
print("Unit_Price min/max:", df["Unit_Price"].min(), "/", df["Unit_Price"].max())
print("Any negative Units_Sold left?:", (df["Units_Sold"] < 0).any())

# Save the cleaned dataset
df.to_csv("cleaned_business_performance_dataset1.csv", index=False)
print("\nSaved cleaned_business_performance_dataset.csv")


 === FINAL VALIDATION ===
Final Shape: (3786, 13)

Remaining nulls per column:
Transaction_ID              0
Order_Date                  0
Region                      0
Product_Category            0
Product_Name                0
Units_Sold                  0
Unit_Price                  0
Total_Revenue               0
Customer_Name               0
Payment_Method              0
Missing_Numeric_Flag        0
Total_Revenue_Calculated    0
Revenue_Mismatch_Flag       0
dtype: int64

 Data types: 
Transaction_ID                       int64
Order_Date                  datetime64[ns]
Region                      string[python]
Product_Category            string[python]
Product_Name                string[python]
Units_Sold                         Float64
Unit_Price                         Float64
Total_Revenue                      Float64
Customer_Name                       object
Payment_Method              string[python]
Missing_Numeric_Flag                  bool
Total_Revenue_Calculated     